# 技能3 · Day 1 上机：用实验基准审计观测因果估计

**版本**：v5.0 + CQ-S3-1 质量补强
**配套**：notes.md（讲义）｜ data/README.md（数据来源）｜ solution.ipynb（参考答案，做完再看）

## 学习目标
学完你能：
1. 区分 `nsw_mixtape` 随机实验样本与 `cps_mixtape` 观测对照，并用实验均值差建立基准
2. 用 DoWhy 完成“建模→识别→估计→反驳”，同时检查 overlap 与识别假设
3. 比较观测朴素值、调整估计和实验基准，并正确解释 refuter 不能证明因果为真

## 说明
本笔记本有 **6 个 TODO**，你需要自己填写代码。每个 TODO 有提示。

## 0. 环境准备
首次运行需安装依赖（取消注释执行一次）：

In [ ]:
# !pip install causaldata dowhy econml -q

## 1. 数据集背景与营销映射

本上机区分 `nsw_mixtape` 随机实验样本与 `cps_mixtape` 观测对照库。你将先用 NSW 实验组内均值差建立基准，再把 NSW 处理组与 CPS 对照组拼成观测样本，检验选择偏差与后门调整。

| 数据变量 | 营销映射 | 角色 |
|---------|---------|------|
| `treat` | 是否收到优惠券/看到广告 | 处理 T |
| `re78` | 转化率 / GMV / 客单价 | 结果 Y |
| `age`,`educ`,`re74`,`re75`,`black`,`hisp`,`marr`,`nodegree` | 用户画像 / 历史消费 | 处理前协变量 X |

**因果问题**：观测样本的朴素均值差与实验基准相差多少？后门调整在什么假设和 overlap 范围内可能缩小差距？

In [ ]:
import pandas as pd
import numpy as np
import dowhy
from dowhy import CausalModel
from causaldata import nsw_mixtape, cps_mixtape

## TODO 1-2：加载与探索真实数据

In [ ]:
# TODO 1：加载实验样本与观测对照，并构造观测比较样本
# 必须实际调用以下两个公开数据模块：
# nsw_mixtape.load_pandas()
# cps_mixtape.load_pandas()
# 要求：rct_df 保留完整 NSW 实验样本；df = NSW处理组 + CPS观测对照

# ===== 你的代码 =====
rct_df = None  # TODO
df = None      # TODO
# ====================

print(f"NSW 实验样本: {rct_df.shape}")
print(f"观测比较样本: {df.shape}")
df.head()

In [ ]:
# TODO 2：探索观测样本的协变量均衡性
# 合法字段："age", "educ", "black", "hisp", "marr", "nodegree", "re74", "re75"
# 要求：打印两组样本量、协变量均值和 SMD，并说明还需检查倾向得分 overlap

# ===== 你的代码 =====

# ====================

## 2. 因果图（DAG）与样本选择

NSW 实验内部的处理是随机分配；把 NSW 处理组与 CPS 观测对照拼接后，样本来源使 X 同时预测组别和结果，形成选择偏差。

```
X ──> sample/treat ──> re78
│                    ▲
└────────────────────┘
```

后门调整需明确 consistency、exchangeability、positivity 和 SUTVA。营销中也不能因为控制了历史活跃度，就假设所有未观测投放规则已经消失。

## TODO 3：朴素估计（有偏）

In [ ]:
# TODO 3：分别计算 NSW 实验基准与观测样本朴素均值差
# 要求：rct_ate 使用 rct_df；naive_ate 使用 df；报告二者差距

# ===== 你的代码 =====
rct_ate = None    # TODO
naive_ate = None  # TODO
# ====================

print(f"NSW 实验基准 = {rct_ate:.2f}")
print(f"观测样本朴素均值差 = {naive_ate:.2f}")

## 3. 为什么观测比较会有偏

偏差来自比较设计，不是“真实数据天然有混杂”。NSW 实验组内均值差是随机实验基准；NSW 处理组与 CPS 对照组来自不同选择机制。后门调整只能在共同支撑和无未观测混杂合理时减少偏差，不能自动证明因果。

## TODO 4-5：DoWhy 因果分析

In [ ]:
# TODO 4：用 DoWhy 建模 → 识别 → 估计（后门调整）
# 提示：common_causes 使用实际字段
# ["age", "educ", "black", "hisp", "marr", "nodegree", "re74", "re75"]
# 要求：打印调整估计、观测朴素值与实验基准，不得把接近基准解释为假设已证明

# ===== 你的代码 =====
model = None                # TODO
identified_estimand = None  # TODO
causal_estimate = None      # TODO
# ====================

print(f"后门调整估计 = {causal_estimate.value:.2f}")
print(f"观测朴素值 {naive_ate:.2f} | 实验基准 {rct_ate:.2f}")

In [ ]:
# TODO 5：反驳检验（安慰剂处理）—— 验证估计稳健性
# 提示：model.refute_estimate(identified_estimand, causal_estimate, "placebo_treatment_refuter")
# 要求：打印安慰剂检验结果，看新估计是否接近 0

# ===== 你的代码 =====
refutation = None  # TODO
# ====================

print(refutation)

## 4. 营销延伸：倾向得分匹配（PSM）

后门调整（线性回归）是一种方法。PSM 是另一种常用的观测数据因果估计法：按"收到处理的概率"（倾向得分）把处理组与对照组匹配，再算匹配后的均值差。

在营销中，PSM 常用于把"收到优惠券的用户"与"相似但没收到优惠券的用户"匹配，估计优惠券的真实增量效应。

In [ ]:
# TODO 6（可选）：用 PSM 再估一次，对比三种估计
# 提示：method_name="backdoor.propensity_score_matching"
# 要求：打印 PSM 估计值，与朴素、后门回归对比

# ===== 你的代码 =====
psm_estimate = None  # TODO
# ====================

print(f"PSM 估计 ATE = {psm_estimate.value:.2f}")
print(f"三种估计对比：朴素 {naive_ate:.2f} | 后门回归 {causal_estimate.value:.2f} | PSM {psm_estimate.value:.2f}")

## 5. 反思与前沿

### 反思问题
1. 朴素估计与后门调整估计的差异，主要来自哪个混杂变量？（提示：看 TODO2 的均衡性对比，哪 个协变量两组差距最大）
2. 安慰剂检验结果是否支持你的因果估计？（若安慰剂效应≈0，说明方法没在虚假处理上"发现"效应，方法可靠）
3. 如果 NSW 数据里有个**没观测到的混杂**（如"个人上进心"），你的估计还可靠吗？→ 这是"可忽略性"假设的根本局限

### 2026 前沿：LLM-as-a-judge 自检因果论证
把你建好的 DAG + 识别策略 + 估计 + 反驳结果整理成一段结构化描述，让 LLM 扮演"因果推断评审"，检查：
- DAG 是否遗漏了可能的混杂？
- 识别策略是否满足后门准则？
- 反驳检验是否充分？
- 结论是否过度外推？

参考 arXiv 2306.05685（NeurIPS 2023, LLM-as-a-judge）。**注意**：LLM 只审查论证质量，不估计效应本身——它停留在因果阶梯 L1，不能上升到 L2/L3。